# 003_analysis

## Purpose

This notebook analyzes the **structure and propagation behavior of shock events** across the crypto universe. 

It consumes the enriched files produced by **002_enrich** and focuses on understanding:

1.  **Synchronous shocks** --- when multiple assets experience shock events on the same day.
2.  **Event propagation** --- how shocks in one asset influence the returns of other assets over subsequent days.

------------------------------------------------------------------------

## Inputs

Artifacts produced by **002_enrich**:

  | Artifact                | Description                                      |
  | ------------------------| -------------------------------------------------|
  | price_wide.pkl          | Wide price table indexed by date                 |
  | returns_full.pkl        | Daily returns for all coins                      |
  | rolling_sigma.pkl       | Rolling volatility estimates                     |
  | z_scores.pkl            | Standardized return shocks                       |
  | events.pkl              | Event list extracted from z-score thresholding   |
  | event_bitmap.pkl        | Binary matrix marking event days by coin         |
  | event_count_matrix.pkl  | Count of detected events                         |

------------------------------------------------------------------------

## Analytical Goals

This notebook produces three main analytical views:

1.  **Pair Co‑Shock Matrix**\
    Measures how often two assets experience events on the same day.

2.  **Normalized Co‑Shock Rates**\
    Estimates conditional probabilities of shocks across asset pairs.

3.  **Event Propagation Dataset**\
    Measures how other assets respond in the **days following a shock event**.

------------------------------------------------------------------------

## Outputs

Artifacts written to:

outputs/003_analysis/

  |Artifact               | Description                         |
  |-----------------------| ------------------------------------|
  | co_shock_matrix.pkl   | Raw counts of simultaneous shocks   |
  | co_shock_rate.pkl     | Normalized co‑shock probabilities   |
  | event_propagation.pkl | Event to future return panel        |
  | lag_response.pkl      | Aggregated lag response statistics  |

------------------------------------------------------------------------

## Design Principle

The research pipeline follows a **layered architecture**:

* 001_download to data acquisition
* 002_enrich to derived signals
* 003_analysis to statistical structure
* 004_signal to trading hypotheses

Each layer consumes artifacts from the previous stage and produces outputs that can be reproduced.

### 1. Imports and Environment Setup
### Provide the necessary imports required for to to proceed.   

In [1]:
import pandas as pd
import numpy as np
import pickle
from datetime import datetime
import math
from pathlib import Path

### 2. Prepare the environment for the notebook

In [2]:
startdate = "2023-01-01"
trading_days = 252
frequency = "1d"

universe = [
    "BTCUSDT",   # Bitcoin
    "ETHUSDT",   # Ethereum
    "BNBUSDT",   # Binance Coin
    "SOLUSDT",   # Solana
    "XRPUSDT",   # Ripple
    "ADAUSDT",   # Cardano
    "DOGEUSDT",  # Dogecoin
    "AVAXUSDT",  # Avalanche
    "LTCUSDT"    # Litecoin
]

execution_delay = [0, 1, 2, 3]
execution_cost_bps = [20, 30, 40]

stage_label = "003_analysis"

OUTPUT_ROOT = Path("../output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MANIFEST_FILE = OUTPUT_ROOT / "manifest.pkl"

DOWNLOAD_DIR = OUTPUT_ROOT / "001_download"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

ENRICH_DIR = OUTPUT_ROOT / "002_enrich"
ENRICH_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_DIR = OUTPUT_ROOT / "003_analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

inspection_window = 20

observation_window_length = 10
observation_window = range(1, observation_window_length + 1)

holding_period = 1


### 2.1 Loading the manifest pickle file

In [3]:
MANIFEST_FILE = OUTPUT_ROOT / "manifest.pkl"

if MANIFEST_FILE.exists():
    manifest = pd.read_pickle(MANIFEST_FILE)
else:
    manifest = {}

manifest.setdefault(stage_label, {})

{}

### 3. Load the pickle files from the previous stage
We use these file contents for our analysis.

In [4]:
EVENTS_RAW_FILE = ENRICH_DIR / "events_raw.pkl"
EVENTS_FILE = ENRICH_DIR / "events.pkl"
PRICES_RAW_FILE = ENRICH_DIR / "prices_raw.pkl"
PRICE_WIDE_FILE = ENRICH_DIR / "price_wide.pkl"
RETURNS_FULL_FILE = ENRICH_DIR / "returns_full.pkl"
ROLLING_SIGMA_FILE = ENRICH_DIR / "rolling_sigma.pkl"
Z_SCORES_FILE = ENRICH_DIR / "z_scores.pkl"

events_raw = pd.read_pickle(EVENTS_RAW_FILE)
events = pd.read_pickle(EVENTS_FILE)
prices_raw = pd.read_pickle(PRICES_RAW_FILE)
price_wide = pd.read_pickle(PRICE_WIDE_FILE)
returns_full = pd.read_pickle(RETURNS_FULL_FILE)
rolling_sigma = pd.read_pickle(ROLLING_SIGMA_FILE)
z_scores = pd.read_pickle(Z_SCORES_FILE)

print("events_raw:", events_raw.shape)
print("events:", events.shape)
print("price_wide:", price_wide.shape)
print("returns_full:", returns_full.shape)
print("rolling_sigma:", rolling_sigma.shape)
print("z_scores:", z_scores.shape)

events_raw: (69480, 5)
events: (69480, 5)
price_wide: (1169, 9)
returns_full: (974, 9)
rolling_sigma: (955, 9)
z_scores: (955, 9)


### 4. Explore sigma thresholds across assets

To understand where statistically significant activity occurs, we iterate over a range of sigma thresholds.

For each threshold we:

1. Apply the sigma filter to the standardized return matrix
2. Select events for a specific coin
3. Inspect the remaining event dates

This allows us to observe how the number of candidate events changes as the statistical threshold increases.

I explored how the number of detected events changes as the sigma threshold increases. 
The goal is to identify a threshold that isolates meaningful market shocks while avoiding excessive noise.
And the point here is to make our analysis of interesting events be based on the sigma so this is not arbitrary.

Observed event counts:

| Sigma Threshold | Event Count | Approx. Frequency |
|-----------------|------------|-------------------|
| ≥ 1σ | 244 | ~25% of days |
| ≥ 2σ | 62 | ~6% of days |
| ≥ 3σ | 14 | ~1.4% of days |
| ≥ 4σ | 3 | ~0.3% of days |
| ≥ 5σ | 0 | none observed |

Dataset size: **971 trading days**

Interpretation:

* **1σ** events occur too frequently and probably normal market noise.
* **2σ** events are fairly common and thus may include both signal and noise.
* **3σ** events appear at a useful cadence ( estimated: 1 event every 70 days) and may represent market shocks.
* **4σ** events are rare ( estimated: 1 per year) and may correspond to major market moves.
* **5σ** events are expected to be significant events, such as Lehman Shock, COVID, etc. They were not observed in the sample.

Conclusion:

For the purposes of event propagation analysis, the most informative trigger range will likely fall between **3σ and 4σ**.

This provides enough observations to analyze cross-asset reactions while still isolating statistically significant market events.

In [5]:
sigma_levels = range(1, 6)

sigma_event_matrix = pd.DataFrame({
    coin: [(z_scores[coin].abs() >= s).sum() for s in sigma_levels]
    for coin in sorted(z_scores.columns)
}).T

sigma_event_matrix.columns = [f"{s}σ" for s in sigma_levels]

# persist artifact
SIGMA_EVENT_MATRIX_FILE = ANALYSIS_DIR / "sigma_event_matrix.pkl"
sigma_event_matrix.to_pickle(SIGMA_EVENT_MATRIX_FILE)

# register artifact in manifest
manifest[stage_label]["sigma_event_matrix"] = str(SIGMA_EVENT_MATRIX_FILE)

sigma_event_matrix

,1σ,2σ,3σ,4σ,5σ
ADAUSDT,269,58,9,1,0
AVAXUSDT,269,58,8,0,0
BNBUSDT,262,74,12,0,0
BTCUSDT,266,66,11,1,0
DOGEUSDT,250,60,11,0,0
ETHUSDT,266,61,13,2,0
LTCUSDT,248,56,15,0,0
SOLUSDT,293,59,8,0,0
XRPUSDT,244,63,17,2,0


### 5. Inspect extreme standardized returns positive and negative into same table

In [6]:
extreme_table = pd.DataFrame({
    "max_z": z_scores.max(),
    "max_date": z_scores.idxmax(),
    "min_z": z_scores.min(),
    "min_date": z_scores.idxmin()
})

# largest absolute shock magnitude
extreme_table["largest_abs"] = extreme_table[["max_z", "min_z"]].abs().max(axis=1)

# deterministic ordering
extreme_table = extreme_table.sort_values("largest_abs", ascending=False)

# persist artifact
EXTREME_Z_FILE = ANALYSIS_DIR / "extreme_z_scores.pkl"
extreme_table.to_pickle(EXTREME_Z_FILE)

# register artifact in manifest
manifest[stage_label]["extreme_z_scores"] = str(EXTREME_Z_FILE)

extreme_table

,max_z,max_date,min_z,min_date,largest_abs
coin,,,,,
ADAUSDT,4.281478,2025-03-02 00:00:00+00:00,-3.731054,2025-10-10 00:00:00+00:00,4.281478
ETHUSDT,4.040426,2023-11-09 00:00:00+00:00,-4.155917,2023-08-17 00:00:00+00:00,4.155917
BTCUSDT,3.741367,2023-10-23 00:00:00+00:00,-4.079907,2023-08-17 00:00:00+00:00,4.079907
XRPUSDT,3.867720,2025-03-02 00:00:00+00:00,-4.044816,2026-02-05 00:00:00+00:00,4.044816
BNBUSDT,3.841387,2024-12-03 00:00:00+00:00,-3.440522,2023-08-17 00:00:00+00:00,3.841387
AVAXUSDT,3.318753,2024-03-11 00:00:00+00:00,-3.811970,2025-10-10 00:00:00+00:00,3.811970
DOGEUSDT,3.805705,2024-02-28 00:00:00+00:00,-3.393044,2025-10-10 00:00:00+00:00,3.805705
LTCUSDT,3.453240,2025-07-19 00:00:00+00:00,-3.745821,2025-10-10 00:00:00+00:00,3.745821
SOLUSDT,3.642370,2023-11-10 00:00:00+00:00,-3.586782,2025-02-24 00:00:00+00:00,3.642370


In [7]:
threshold = 3
event_density = (z_scores.abs() >= threshold).sum(axis=1)
event_density.sort_values(ascending=False).head(15)

date
2026-02-05 00:00:00+00:00    9
2023-08-17 00:00:00+00:00    7
2025-05-08 00:00:00+00:00    6
2025-10-10 00:00:00+00:00    5
2025-02-24 00:00:00+00:00    4
2025-03-02 00:00:00+00:00    4
2024-07-04 00:00:00+00:00    4
2025-04-06 00:00:00+00:00    3
2024-03-11 00:00:00+00:00    3
2023-11-10 00:00:00+00:00    2
2025-01-15 00:00:00+00:00    2
2023-10-23 00:00:00+00:00    2
2024-02-28 00:00:00+00:00    2
2024-11-06 00:00:00+00:00    2
2024-05-20 00:00:00+00:00    2
dtype: int64

In [8]:
threshold = 3
rows = []

for date, row in z_scores.iterrows():

    shock_mask = row.abs() >= threshold

    if shock_mask.sum() >= 2:

        record = {"date": date, "count": int(shock_mask.sum())}

        for coin in z_scores.columns:
            if shock_mask[coin]:
                record[coin] = "+" if row[coin] > 0 else "-"
            else:
                record[coin] = "."

        rows.append(record)

event_bitmap = pd.DataFrame(rows)

# deterministic column order
event_bitmap = event_bitmap[
    ["date", "count"] + sorted(z_scores.columns)
]

# sort by largest cross-asset shock
event_bitmap = event_bitmap.sort_values("count", ascending=False)

# persist artifact
EVENT_BITMAP_FILE = ANALYSIS_DIR / "event_bitmap.pkl"
event_bitmap.to_pickle(EVENT_BITMAP_FILE)

# register artifact
manifest[stage_label]["event_bitmap"] = str(EVENT_BITMAP_FILE)

event_bitmap.head(20)

,date,count,ADAUSDT,AVAXUSDT,BNBUSDT,BTCUSDT,DOGEUSDT,ETHUSDT,LTCUSDT,SOLUSDT,XRPUSDT
18,2026-02-05 00:00:00+00:00,9,-,-,-,-,-,-,-,-,-
0,2023-08-17 00:00:00+00:00,7,-,-,-,-,.,-,-,.,-
14,2025-05-08 00:00:00+00:00,6,+,+,.,.,+,+,.,+,+
15,2025-10-10 00:00:00+00:00,5,-,-,.,.,-,.,-,.,-
12,2025-03-02 00:00:00+00:00,4,+,.,.,+,.,.,.,+,+
7,2024-07-04 00:00:00+00:00,4,-,.,-,.,.,-,.,.,-
11,2025-02-24 00:00:00+00:00,4,.,.,.,-,-,-,.,-,.
4,2024-03-11 00:00:00+00:00,3,.,+,.,.,.,.,+,.,+
13,2025-04-06 00:00:00+00:00,3,-,.,.,.,.,-,-,.,.
2,2023-11-10 00:00:00+00:00,2,.,+,.,.,.,.,.,+,.


### 999. Persist the information to the pickle file

In [9]:
pd.to_pickle(manifest, MANIFEST_FILE)
print("manifest saved:", MANIFEST_FILE)
print("pipeline stages:", sorted(manifest.keys()))
print(f"artifacts in {stage_label}:", sorted(manifest[stage_label].keys()))

manifest saved: ../output/manifest.pkl
pipeline stages: ['001_download', '002_enrich', '003_analysis']
artifacts in 003_analysis: ['event_bitmap', 'extreme_z_scores', 'sigma_event_matrix']
